In [13]:
import tiktoken
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv

partie LLM

In [14]:
load_dotenv(override=True)

True

In [15]:
from langchain_openai import ChatOpenAI
llm= ChatOpenAI(model="gpt-4o-mini",temperature=0.0)

1) Implémentation du Rag

1.1) load le pdf

In [27]:
pdf_file = "C:/iaagentique/TP2/pdf/Rapport.pdf"

PyPDFLoader

In [28]:
pdf_loader = PyPDFLoader(pdf_file)

In [42]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

In [43]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [48]:
chunks = loader.load_and_split(text_splitter)


ImportError: `pypdf` package not found, please install it with `pip install pypdf`

In [45]:
len(chunks)

0

CHROMADB,EMBEDDING pour la sémantique et le stockage vectoriel

In [22]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

Construction du vector store pour stocker les embedings/sémantique dans chromadb

In [25]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="rapport_ocp_V2",
    persist_directory="./store"
)

ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

on crée un retriever pour récuperer du texte la partie la plus proche sémantiquement de notre question

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

exemple de questions

In [ ]:
retrieved_chunks=retriever.invoke("Quelles sont les performances financières de l'OCP en 2023?")

NameError: name 'retriever' is not defined

2) Partie RAG:

In [ ]:
prompt_template = """
Answer the following question based only on provided context and in the form  of a suprsingly whimsical poem if possible
The context is about OCP Annual Financial Report 2023
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS, pick up a book and see if you can do better
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

On récupère les docs nécessaire

In [ ]:
user_input = "Quelles sont les performances financières de l'OCP en 2023?"
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)
context_for_query

construction du prompt

In [ ]:
prompt = prompt_template.format(context=context_for_query, question=user_input)
print(prompt)


réponse au prompt

In [ ]:
from IPython.display import Markdown
resp = llm.invoke(prompt)
print(display(Markdown(resp.content)))

NameError: name 'llm' is not defined

On définit une fonction pour répondre au prompt

In [ ]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [ ]:
response = RAG("État du résultat global consolidé")
print(display(Markdown(response)))

exemple d'un hors sujet

In [ ]:
user_input = "pourquoi les magasins ouvert H24 ont t'ils des verrous ?"
output = RAG(user_input)
print(output)

exemple d'un ...bah un in sujet i guess

In [ ]:
response = RAG("Quelles sont les performances financières de l'OCP en 2023")
print(display(Markdown(response)))

3) Evaluation LLM AS a judge

In [ ]:
user_input ="État du résultat global consolidé"


relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)


user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [ ]:
# Default answer for an RAG query
answer = RAG(user_input)
print(display(Markdown(answer)))

A) respect de la métrique

In [ ]:
groundedness_rater_system_message="""
Vous êtes chargé d’évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d’IA pour générer la réponse, ainsi qu’une réponse générée par l’IA à la question.

Dans l’entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l’IA commencera par ###Answer.

Critères d’évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 — La métrique n’est pas respectée du tout
2 — La métrique n’est respectée que dans une mesure limitée
3 — La métrique est respectée dans une bonne mesure
4 — La métrique est respectée en grande partie
5 — La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions :

Écrivez d’abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d’évaluation et attribuer un score.
"""

In [ ]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [ ]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


In [ ]:
resp=evaluate(groundedness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

PARTIE 2 :